# 已失败分钟turnover来源：同bar/相邻bar口径只读诊断

原计划的同一分钟VWAP落在原价OHLC范围≥99.9%门已失败。本卡维持1bp容忍，仅比较报告`turnover/volume`与同bar、前一bar、后一bar的原价高低范围，严格在同一交易日内移动范围；数值只定位源语义，不把邻bar事后匹配当作新可用特征，也不读取未来收益。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
root=Path.cwd();card=root/'research/stage1_v5_raw_turnover_semantics_audit_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['failed_source_card']/'plan.json')==plan['failed_source_plan_sha256'] and sha(root/plan['failed_source_card']/'audit.json')==plan['failed_source_audit_sha256']
assert sha(root/plan['source_manifest'])==plan['source_manifest_sha256'] and sha(root/plan['nas_binding'])==plan['nas_binding_sha256'] and sha(root/plan['qlib_provider_uri']/'qlib_store_manifest.json')==plan['qlib_provider_manifest_sha256']
qlib.init(provider_uri=str(root/plan['qlib_provider_uri']),region='us',kernels=1)
raw={}
for symbol,spec in plan['original_parquets'].items():
 path=root/spec['relative_path'];assert sha(path)==spec['sha256']
 x=pd.read_parquet(path,columns=['time_key','open','high','low','close','volume','turnover']);x['datetime']=pd.to_datetime(x.time_key)
 x=x.loc[x.datetime.between('2018-10-15 09:31:00','2022-12-22 16:00:00')].set_index('datetime').sort_index();raw[symbol]=x
print('original bytes/manifest identities passed')

[44091:MainThread](2026-10-30 16:18:28,340) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[44091:MainThread](2026-10-30 16:18:28,701) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[44091:MainThread](2026-10-30 16:18:28,701) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


original bytes/manifest identities passed


In [2]:
bars=D.features(['SOXL','SOXX'],['$raw_high','$raw_low','$raw_close','$raw_volume','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min');assert len(bars)==820440
rows=[];examples=[]
for symbol in ('SOXL','SOXX'):
 q=bars.xs(symbol,level='instrument');source=raw[symbol];assert q.index.equals(source.index)
 day=source.index.normalize();price=source.turnover.to_numpy(float)/source.volume.replace(0,np.nan).to_numpy(float)
 base=source.copy();base['vwap']=price;base['valid']=q['$price_valid'].eq(1).to_numpy()
 positive=base.valid&base.volume.gt(0)&base.turnover.gt(0)
 for shift in plan['shifts_in_completed_bars']:
  low=base.low.groupby(day).shift(shift);high=base.high.groupby(day).shift(shift);neighbor_valid=base.valid.groupby(day).shift(shift).fillna(False).astype(bool)
  mask=positive&neighbor_valid&low.notna()&high.notna()
  inside=(base.vwap>=low*np.exp(-plan['tolerance_bps']/10000))&(base.vwap<=high*np.exp(plan['tolerance_bps']/10000))
  outside=np.where(base.vwap<low,10000*np.log(low/base.vwap),np.where(base.vwap>high,10000*np.log(base.vwap/high),0.))
  for year in (2018,2019,2020,2021,2022):
   y=mask&(base.index.year==year)
   d=base.index[y].normalize()
   rows.append({'symbol':symbol,'year':year,'range_shift':shift,'bars':int(y.sum()),'days':int(d.nunique()),'inside_fraction_1bp':float(inside[y].mean()),'outside_median_distance_bp':float(np.median(outside[y&~inside])) if (y&~inside).any() else 0.,'outside_p90_distance_bp':float(np.quantile(outside[y&~inside],.9)) if (y&~inside).any() else 0.})
  if shift==0:
   bad=base.loc[mask&~inside].head(3)
   for stamp,b in bad.iterrows():examples.append({'symbol':symbol,'bar_end_at':str(stamp),'raw_low':float(b.low),'raw_high':float(b.high),'raw_close':float(b.close),'volume':float(b.volume),'turnover':float(b.turnover),'implied_vwap':float(b.vwap)})
result=pd.DataFrame(rows).sort_values(['symbol','year','range_shift']);result.to_csv(card/'shifted_range_summary.csv',index=False)
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'source_pair_rows':len(bars),'same_bar_inside_fraction':float(result.loc[result.range_shift.eq(0)].bars.mul(result.loc[result.range_shift.eq(0),'inside_fraction_1bp']).sum()/result.loc[result.range_shift.eq(0),'bars'].sum()),'shifted_range_summary':result.to_dict('records'),'first_same_bar_outside_examples':examples,'failed_original_gate_remains_failed':True,'future_economic_values_accessed':False,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'same_bar':audit['same_bar_inside_fraction'],'examples':examples,'summary':{str(k):float(v) for k,v in result.groupby(['symbol','range_shift']).apply(lambda g:float((g.bars*g.inside_fraction_1bp).sum()/g.bars.sum())).items()}},ensure_ascii=False,indent=2))

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_44091/1006681563.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  low=base.low.groupby(day).shift(shift);high=base.high.groupby(day).shift(shift);neighbor_valid=base.valid.groupby(day).shift(shift).fillna(False).astype(bool)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_44091/1006681563.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  low=base.low.groupby(day).shift(shift);high=base.high.groupby(day).shift(shift);neighbor_valid=base.valid.groupby(day).shift(shift).fillna(False)

{
  "same_bar": 0.9077165427805977,
  "examples": [
    {
      "symbol": "SOXL",
      "bar_end_at": "2018-10-15 09:37:00",
      "raw_low": 115.12,
      "raw_high": 115.12,
      "raw_close": 115.12,
      "volume": 126.0,
      "turnover": 14543.68,
      "implied_vwap": 115.42603174603175
    },
    {
      "symbol": "SOXL",
      "bar_end_at": "2018-10-15 09:45:00",
      "raw_low": 114.85,
      "raw_high": 115.05,
      "raw_close": 115.021,
      "volume": 2187.0,
      "turnover": 251688.383,
      "implied_vwap": 115.08385139460448
    },
    {
      "symbol": "SOXL",
      "bar_end_at": "2018-10-15 09:52:00",
      "raw_low": 116.0,
      "raw_high": 116.0,
      "raw_close": 116.0,
      "volume": 666.0,
      "turnover": 77215.2,
      "implied_vwap": 115.93873873873873
    },
    {
      "symbol": "SOXX",
      "bar_end_at": "2018-10-15 09:49:00",
      "raw_low": 168.93,
      "raw_high": 168.95,
      "raw_close": 168.95,
      "volume": 336.0,
      "turnover": 56773.

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_44091/1006681563.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  low=base.low.groupby(day).shift(shift);high=base.high.groupby(day).shift(shift);neighbor_valid=base.valid.groupby(day).shift(shift).fillna(False).astype(bool)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_44091/1006681563.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(json.dumps({'same_bar':audit['same_bar_inside_fraction'],'examples':

## 中文结论

同bar放宽1bp的隐含VWAP落入率90.77%；对前/后bar高低区间仅约45%–51%，简单时间戳错位不能解释源门失败。低成交量且高低价相同的bar常见数bp越界；原始turnover与Qlib volume时间已对齐。此只读诊断不修改原99.9%来源门、不移动时间戳，也没有模型/账户。